# 1. Wildfire Classification — Baseline Model

This notebook establishes the baseline machine learning model for wildfire classification using the WildFires_DataSet_a_interpoler.csv dataset.

The objective is to predict whether an observation belongs to the fire or no_fire class using environmental features such as NDVI, LST, and BURNED_AREA.

This notebook represents the initial, notebook-based implementation of the machine learning workflow. It will later serve as the reference implementation when the model is transformed into a production-ready MLOps service.

# 2. Problem Definition

### 2.1 Business / ML Problem

Wildfires can cause significant environmental and economic damage. Early identification of wildfire-related conditions can support monitoring and decision-making systems.

In this project, we use environmental features extracted from wildfire-related observations to build a machine learning model that classifies each observation as either `fire` or `no_fire`.

The objective of this baseline project is not to build the most accurate wildfire detection system, but to establish a simple, measurable machine learning baseline that can later be transformed into a production-ready MLOps system.


### 2.2 Objective

The objective is to develop a binary classification model that predicts whether an observation belongs to the `fire` or `no_fire` class using the available environmental features.

The baseline model will provide a reference point for the later stages of the MLOps project, where the model will be packaged, tested, served through an API, containerized, and monitored.


### 2.3 Target Variable

The target variable is `CLASS`.

It contains two classes:

- `fire`
- `no_fire`

This makes the task a binary classification problem.

For model training, the target will later be encoded numerically:

- `no_fire` → 0
- `fire` → 1


### 2.4 Input Features

The model will use the following environmental features:

| Feature | Description |
|---|---|
| `NDVI` | Normalized Difference Vegetation Index |
| `LST` | Land Surface Temperature |
| `BURNED_AREA` | Burned area measurement |

These features will be investigated during exploratory data analysis to understand their distributions, relationships, missing values, and potential usefulness for classification.


### 2.5 Problem Type

This is a supervised binary classification problem.

- **Learning type:** Supervised learning
- **Problem type:** Binary classification
- **Target:** `CLASS`
- **Classes:** `fire`, `no_fire`
- **Features:** `NDVI`, `LST`, `BURNED_AREA`


### 2.6 Success Criteria

The baseline model should:

1. Successfully train on the available dataset.
2. Produce predictions for unseen validation data.
3. Achieve measurable performance using accuracy, precision, recall, F1-score, and ROC-AUC.
4. Produce a confusion matrix for error analysis.
5. Establish a reproducible baseline that can be compared with future models.
6. Be saved as a model artifact that can later be integrated into the production ML pipeline.

The baseline metrics will be recorded after model evaluation and will serve as the reference point for future improvements.


# 3. Dataset Overview


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


### 3.1 Load Dataset

The wildfire dataset is stored in a CSV file named `WildFires_DataSet_a_interpoler.csv`.

The dataset will be loaded into a pandas DataFrame for inspection, exploration, preprocessing, and model development.


In [ ]:
DATA_PATH = "/content/WildFires_DataSet_a_interpoler.csv"

df = pd.read_csv(DATA_PATH,sep=";")

df.head()

### 3.2 Dataset Shape

We inspect the shape of the dataset to determine the number of observations and features available for model development.


In [ ]:
#number of row and coloumn
df.shape

### 3.3 Basic Dataset Information

We inspect the overall structure of the dataset, including the number of entries, columns, non-null values, and memory usage.


In [ ]:
#Get basic info about the dataset
df.info()

### 3.4 Data Types

We inspect the data type of each column to understand how the features and target variable are represented and to identify any columns that may require type conversion during preprocessing.


In [ ]:
# Check the data type of each column
df.dtypes


In [ ]:
# Display columns grouped by data type
df.dtypes.value_counts().plot(
    kind="bar",
    color=["steelblue", "orange"]
)

plt.title("Number of Columns by Data Type")
plt.xlabel("Data Type")
plt.ylabel("Number of Columns")
plt.show()


### 3.5 Unique Values

We inspect the unique values in each column to understand the categories present in the dataset and identify any unexpected values.


In [ ]:
# Check the number of unique values in each column
df.nunique()


In [ ]:
# Check the unique classes in the target variable
df["CLASS"].unique()

### 3.6 Missing Values

We inspect the dataset for missing values in each column to understand the extent of missing data before deciding how it should be handled during preprocessing.


In [ ]:
# Count missing values in each column
df.isnull().sum()

In [ ]:
# Calculate the percentage of missing values in each column
df.isnull().mean() * 100


In [ ]:
# Visualize missing values
missing_values = df.isnull().sum()

missing_values[missing_values > 0].plot(
    kind="bar",
    color="tomato"
)

plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=0)
plt.show()


### 3.7 Duplicate Records

We inspect the dataset for duplicate rows to identify repeated observations that may affect model training and evaluation.


In [ ]:
# Count duplicate rows
df.duplicated().sum()


In [ ]:
# Display duplicate rows
df[df.duplicated()].head(5)


### 3.8 Dataset Summary

We generate descriptive statistics for the numerical features to understand their central tendency, spread, and value ranges before performing exploratory data analysis.


In [ ]:
# Get summary statistics for numerical columns
df.describe()


In [ ]:
# Create a compact dataset summary
summary = pd.DataFrame({
    "Rows": [df.shape[0]],
    "Columns": [df.shape[1]],
    "Missing Values": [df.isnull().sum().sum()],
    "Duplicate Rows": [df.duplicated().sum()],
    "Target Classes": [df["CLASS"].nunique()]
})

summary


In [ ]:
# Display a compact overview of all columns
overview = pd.DataFrame({
    "Data Type": df.dtypes,
    "Unique Values": df.nunique(),
    "Missing Values": df.isnull().sum()
})

overview


# 4. Exploratory Data Analysis




### 4.1 Target Distribution

We analyze the distribution of the target variable, `CLASS`, to understand the balance between fire and no-fire observations. This is important because class imbalance can affect model training and evaluation.



In [ ]:
# Count observations in each class
class_counts = df["CLASS"].value_counts()

class_counts


In [ ]:
# Plot target class distribution
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="CLASS",
    hue="CLASS",
    palette="Set2",
    legend=False
)

plt.title("Wildfire Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Observations")

plt.show()


In [ ]:
# Calculate class percentages
class_percentage = df["CLASS"].value_counts(normalize=True) * 100

class_percentage


In [ ]:
# Plot class percentages
plt.figure(figsize=(7, 5))

class_percentage.plot(
    kind="bar",
    color=["steelblue", "tomato"]
)

plt.title("Wildfire Class Distribution (%)")
plt.xlabel("Class")
plt.ylabel("Percentage")

plt.xticks(rotation=0)
plt.show()


### 4.2 Feature Distributions

We visualize the distributions of the numerical features to understand their ranges, shapes, skewness, concentration of values, and potential unusual observations.


In [ ]:
# Plot distributions of numerical features
df[["NDVI", "LST", "BURNED_AREA"]].hist(figsize=(14, 6))


In [ ]:
# Plot distributions of numerical features
numerical_features = ["NDVI", "LST", "BURNED_AREA"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, column in zip(axes, numerical_features):
    sns.histplot(
        data=df,
        x=column,
        kde=True,
        ax=ax,
        color="steelblue"
    )

    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
# Plot box plots for numerical features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, column in zip(axes, numerical_features):
    sns.boxplot(
        data=df,
        y=column,
        ax=ax,
        color="lightblue"
    )

    ax.set_title(f"Box Plot of {column}")
    ax.set_ylabel(column)

plt.tight_layout()
plt.show()


### 4.3 Feature Statistics

We calculate descriptive statistics for the numerical features to understand their central tendency, variability, and value ranges.


In [ ]:
# Calculate descriptive statistics
feature_stats = df[numerical_features].describe().T

feature_stats


In [ ]:
# Add median and skewness
feature_stats["median"] = df[numerical_features].median()
feature_stats["skewness"] = df[numerical_features].skew()

feature_stats


### 4.4 Feature Relationships

We analyze relationships between numerical features using correlation analysis and pairwise visualizations. The goal is to identify potentially related features, linear associations, and visible patterns.


In [ ]:
# Calculate correlation matrix
correlation_matrix = df[numerical_features].corr()

correlation_matrix


In [ ]:
# Visualize feature correlations
plt.figure(figsize=(7, 5))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
# Visualize pairwise relationships
sns.pairplot(
    df[numerical_features],
    diag_kind="hist"
)

plt.show()


### 4.5 Features vs Target

We compare the numerical feature distributions across the fire and no-fire classes to identify differences that may help distinguish the target classes.


In [ ]:
# Compare feature distributions by class
df.groupby("CLASS")[["NDVI", "LST", "BURNED_AREA"]].mean()


In [ ]:
# Compare numerical features across target classes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, column in zip(axes, numerical_features):
    sns.boxplot(
        data=df,
        x="CLASS",
        y=column,
        hue="CLASS",
        palette="Set2",
        legend=False,
        ax=ax
    )

    ax.set_title(f"{column} by Class")
    ax.set_xlabel("Class")
    ax.set_ylabel(column)

plt.tight_layout()
plt.show()


In [ ]:
# Calculate feature means by target class
class_means = df.groupby("CLASS")[numerical_features].mean()

class_means


In [ ]:
# Visualize average feature values by class
class_means.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Average Feature Values by Class")
plt.xlabel("Class")
plt.ylabel("Mean Value")
plt.xticks(rotation=0)

plt.legend(title="Feature")
plt.tight_layout()
plt.show()


### 4.6 Outlier Analysis

We investigate potential outliers in the numerical features using box plots and the Interquartile Range (IQR) method. Potential outliers are examined as observations rather than automatically treated as errors.


In [ ]:
# Count potential outliers using the IQR method
outlier_counts = {}

for column in numerical_features:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_counts[column] = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ).sum()

outlier_counts


In [ ]:
# Visualize outlier counts
plt.figure(figsize=(8, 5))

sns.barplot(
    x=list(outlier_counts.keys()),
    y=list(outlier_counts.values()),
    hue=list(outlier_counts.keys()),
    palette="Set2",
    legend=False
)

plt.title("Potential Outliers by Feature")
plt.xlabel("Feature")
plt.ylabel("Number of Potential Outliers")

plt.show()


### 4.7 EDA Findings

The exploratory data analysis provided the following observations:

- The target variable contains two classes: `fire` and `no_fire`.
- The dataset is imbalanced, with `no_fire` representing approximately **77.47%** of observations and `fire` approximately **22.53%**.
- `NDVI` has a mean of approximately **0.555** and shows a relatively strong negative skewness (**-1.28**), indicating that its distribution is not symmetric.
- `LST` has a mean of approximately **14620.88** and a moderate negative skewness (**-0.40**), with values ranging from approximately **13137 to 15611.57**.
- `BURNED_AREA` has a mean of approximately **4.67** and negative skewness (**-0.87**). It also contains the largest number of IQR-based potential outliers, with **153 observations** identified.
- The numerical features show relatively weak linear correlations with each other. The strongest correlation is between `LST` and `BURNED_AREA` (**0.158**), which indicates a weak positive linear relationship.
- The average `LST` is higher for `fire` observations (**14818.07**) than for `no_fire` observations (**14563.10**).
- The average `NDVI` is slightly lower for `fire` observations (**0.534**) than for `no_fire` observations (**0.562**).
- The average `BURNED_AREA` is slightly higher for `fire` observations (**4.760**) than for `no_fire` observations (**4.650**).
- These differences suggest that `NDVI`, `LST`, and `BURNED_AREA` may contain predictive information for distinguishing between `fire` and `no_fire` observations. However, differences in class means alone are not sufficient to determine predictive performance.
- The detected outliers will not automatically be removed during preprocessing because extreme environmental measurements may represent genuine wildfire conditions rather than data errors.
- Missing values and duplicate records were identified during dataset inspection and will be addressed during the preprocessing stage.
- Because the target variable is imbalanced, accuracy alone should not be considered sufficient for evaluating the baseline classifier. Precision, recall, F1-score, and ROC-AUC will also be considered.
- Overall, the EDA results provide the evidence needed to define the preprocessing strategy and establish the baseline classification model.


# 5. Data Preprocessing

### 5.1 Define Features and Target

The dataset is a binary classification problem where `CLASS` is the target variable.

The numerical columns `NDVI`, `LST`, and `BURNED_AREA` are selected as input features.

We separate the input features from the target before applying preprocessing operations.

In [ ]:
# Define input features and target
features = ["NDVI", "LST", "BURNED_AREA"]
target = "CLASS"

X = df[features].copy()
y = df[target].copy()

# Display the shapes
print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
# Display target values
y.value_counts()


### 5.2 Handle Missing Values

Missing values were identified during the dataset inspection.

Instead of removing observations containing missing values, we will investigate the missing-data pattern and use interpolation for the numerical features.

This approach preserves the available observations while estimating missing numerical values from neighboring observations.


#### 5.2.1 Missing Values Before Interpolation

In [ ]:
# Check missing values before interpolation
missing_before = df[features].isnull().sum()

missing_before


#### 5.2.2 Missing Values Summary

We calculate both the number and percentage of missing values for each column to understand the extent of missing data before applying interpolation.


In [ ]:
import pandas as pd


def missing_values_table(df):
    # Total missing values
    mis_val = df.isnull().sum()

    # Percentage of missing values
    mis_val_percent = 100 * df.isnull().sum() / len(df)

    # Create summary table
    mis_val_table = pd.concat(
        [mis_val, mis_val_percent],
        axis=1
    )

    # Rename columns
    mis_val_table = mis_val_table.rename(
        columns={
            0: "Missing Values",
            1: "% of Total Values"
        }
    )

    # Keep only columns with missing values
    mis_val_table = mis_val_table[
        mis_val_table["% of Total Values"] != 0
    ].sort_values(
        "% of Total Values",
        ascending=False
    ).round(2)

    print(
        f"Dataset contains {df.shape[1]} columns.\n"
        f"{mis_val_table.shape[0]} columns contain missing values."
    )

    return mis_val_table


In [ ]:
# Display missing-value summary
missing_values_table(df[features])


#### 5.2.3 Visualize Missing Data

The missing-data pattern is visualized to determine how missing observations are distributed across the numerical features.

The matrix visualization allows us to inspect whether missing values appear randomly or follow visible patterns.


In [ ]:
import matplotlib.pyplot as plt
import missingno as msno

# Visualize missing-value pattern
msno.matrix(df[features])

plt.title("Missing Data Pattern")
plt.show()


#### 5.2.4 Missingness Correlation

In [ ]:
# Visualize relationships between missing-value patterns
msno.dendrogram(df[features])

plt.title("Missing Data Correlation")
plt.show()


#### 5.2.5 Interpolate Missing Numerical Values

### Linear Interpolation

Missing values in the numerical features are filled using linear interpolation.

Linear interpolation estimates a missing value using neighboring observed values while preserving the existing row order.

The target variable `CLASS` is not interpolated.


In [ ]:
# Create a copy before interpolation
df_interpolated = df.copy()

# Interpolate numerical features
df_interpolated[features] = (
    df_interpolated[features]
    .interpolate(method="linear")
)

#### 5.2.6 Compare Before vs After

#### Missing Values Before and After Interpolation

We compare the number of missing values before and after interpolation to verify that the preprocessing step had the intended effect.


In [ ]:
# Compare missing values before and after interpolation
missing_comparison = pd.DataFrame({
    "Before Interpolation": df[features].isnull().sum(),
    "After Interpolation": df_interpolated[features].isnull().sum()
})

missing_comparison


#### 5.2.7 Validate the Effect on Feature Correlations

#### Correlation Before and After Interpolation

The correlation structure of the numerical features is compared before and after interpolation.

This provides a basic sanity check to determine whether the interpolation process substantially changed the relationships between the features.


In [ ]:
# Correlation before interpolation
correlation_before = df[features].corr()

# Correlation after interpolation
correlation_after = df_interpolated[features].corr()

print("Correlation before interpolation:")
display(correlation_before)

print("Correlation after interpolation:")
display(correlation_after)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before interpolation
sns.heatmap(
    correlation_before,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    ax=axes[0]
)
axes[0].set_title("Correlation Before Interpolation")

# After interpolation
sns.heatmap(
    correlation_after,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    ax=axes[1]
)
axes[1].set_title("Correlation After Interpolation")

plt.tight_layout()
plt.show()


In [ ]:
# Compare distributions before and after interpolation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, feature in enumerate(features):
    sns.kdeplot(
        df[feature].dropna(),
        label="Before",
        ax=axes[i]
    )

    sns.kdeplot(
        df_interpolated[feature].dropna(),
        label="After",
        ax=axes[i]
    )

    axes[i].set_title(feature)
    axes[i].legend()

plt.tight_layout()
plt.show()


### 5.3 Handle Duplicate Records

Duplicate records were identified during the dataset inspection.

A duplicate is considered a row where all selected columns have exactly the same values.

We will remove exact duplicate observations before model training to prevent identical records from receiving unnecessary weight during model fitting.


In [ ]:
# Count exact duplicate rows
duplicate_count = df_interpolated.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)


Remove Exact Duplicate Records

Exact duplicate rows are removed while keeping the first occurrence of each unique observation.


In [ ]:
# Remove exact duplicate rows
df_clean = df_interpolated.drop_duplicates().copy()

print("Original shape:", df_interpolated.shape)
print("Shape after removing duplicates:", df_clean.shape)


In [ ]:
# Verify that no exact duplicates remain
remaining_duplicates = df_clean.duplicated().sum()

print("Remaining duplicate rows:", remaining_duplicates)


In [ ]:
# Calculate number of removed duplicate rows
duplicates_removed = len(df_interpolated) - len(df_clean)

print("Duplicate rows removed:", duplicates_removed)


In [ ]:
print("Original shape:", df_interpolated.shape)
print("Shape after removing duplicates:", df_clean.shape)
print("Duplicate rows removed:", duplicates_removed)


In [ ]:
# Final validation of the clean dataset

expected_columns = ["NDVI", "LST", "BURNED_AREA", "CLASS"]
numerical_columns = ["NDVI", "LST", "BURNED_AREA"]

print("Dataset shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nExpected columns present:",
      list(df_clean.columns) == expected_columns)

print("\nInfinite values:")
print(np.isinf(df_clean[numerical_columns]).sum())

print("\nNegative values:")
print((df_clean[numerical_columns] < 0).sum())

print("\nTarget values:")
print(df_clean["CLASS"].value_counts())

print("\nFinal validation completed.")


In [ ]:
# Create directory for processed data
import os

os.makedirs("/content/data/processed", exist_ok=True)

# Save the clean dataset
clean_data_path = "/content/data/processed/wildfire_data.csv"

df_clean.to_csv(clean_data_path, index=False)

print(f"Clean dataset saved to: {clean_data_path}")


## 5.4 Validate Feature Values

After handling missing values and removing duplicate records, the numerical features are validated to ensure that the preprocessing process did not introduce invalid or unrealistic values.

The validation includes:

- Checking minimum and maximum values.
- Checking for infinite values.
- Checking for negative values where they are not physically meaningful.
- Inspecting the final feature distributions.

These checks provide evidence that the cleaned dataset is suitable for the next preprocessing steps.


In [ ]:
df_clean[features].agg(["min", "max"])


In [ ]:
feature_ranges = df_clean[features].agg(["min", "max"]).T

feature_ranges


In [ ]:
print("Infinite values:")
print(np.isinf(df_clean[features]).sum())


print("Negative values:")
print((df_clean[features] < 0).sum())



In [ ]:
validation_summary = pd.DataFrame({
    "Missing Values": df_clean[features].isnull().sum(),
    "Infinite Values": np.isinf(df_clean[features]).sum(),
    "Minimum": df_clean[features].min(),
    "Maximum": df_clean[features].max()
})

validation_summary


In [ ]:
df_clean[features].hist(
    figsize=(12, 4),
    bins=30,
    edgecolor="black"
)

plt.suptitle(
    "Feature Distributions After Preprocessing"
)

plt.tight_layout()
plt.show()


### 5.5 Encode Target Variable

The target variable `CLASS` contains categorical labels representing whether a wildfire is present.

We encode the target into binary numerical values so that it can be used by the classification model. The `fire` class is encoded as 1 and the `no_fire` class as 0.


In [ ]:
# Encode target classes
df_clean["CLASS"] = df_clean["CLASS"].map({
    "no_fire": 0,
    "fire": 1
})


In [ ]:
# Check encoded target distribution
df_clean["CLASS"].value_counts()


In [ ]:
# Display the unique encoded target values
df_clean["CLASS"].unique()


In [ ]:
# Check for values that were not successfully encoded
missing_encoded_values = y.isnull().sum()

print(
    "Number of target values that failed to encode:",
    missing_encoded_values
)


### 5.6 Feature Transformation

The numerical features are inspected for differences in scale and distribution before transformation. Their descriptive statistics and skewness are evaluated to identify potential scaling or distribution issues.

Standardization using `StandardScaler` is selected as the baseline feature transformation. The scaler will be fitted only on the training data after the train-validation split to prevent data leakage. The same fitted scaler will then be applied to the validation data.


In [ ]:
# ============================================
# 5.6 Feature Transformation
# ============================================

from sklearn.preprocessing import StandardScaler

# Feature statistics before transformation
feature_summary = X[features].agg(
    ["mean", "std", "min", "max"]
).T

print("Feature statistics before transformation:")
display(feature_summary)

# Check feature skewness
skewness = X[features].skew().sort_values(
    ascending=False
)

print("\nFeature skewness:")
display(skewness.to_frame("Skewness"))

# Visualize feature distributions
X[features].hist(
    figsize=(12, 4),
    bins=30,
    edgecolor="black"
)

plt.suptitle("Feature Distributions Before Transformation")
plt.tight_layout()
plt.show()

# Initialize StandardScaler
# It will be fitted only on the training data
# after the train/validation split.
scaler = StandardScaler()

print("StandardScaler initialized successfully.")


### 5.7 Final Dataset Validation

Before creating the training and validation sets, the preprocessed dataset is validated to ensure that it contains no remaining missing values, duplicate records, infinite values, or invalid target values.

This final validation confirms that the dataset is ready for model development.


In [ ]:
# ============================================
# 5.7 Final Dataset Validation
# ============================================

# Use the cleaned dataset
X_final = df_clean[features].copy()
y_final = df_clean[target].copy()

# 1. Check missing values
print("Missing values:")
display(X_final.isnull().sum())

# 2. Check duplicate rows
print("\nDuplicate rows:", X_final.duplicated().sum())

# 3. Check infinite values
print("\nInfinite values:")
print(np.isinf(X_final).sum())

# 4. Check data types
print("\nFeature data types:")
print(X_final.dtypes)

# 5. Check target values
print("\nTarget values:")
print(y_final.value_counts())

# 6. Check dimensions
print("\nDataset dimensions:")
print("Features:", X_final.shape)
print("Target:", y_final.shape)

# 7. Final validation
assert X_final.isnull().sum().sum() == 0, "Missing values remain."
assert X_final.duplicated().sum() == 0, "Duplicate rows remain."
assert np.isinf(X_final).sum().sum() == 0, "Infinite values remain."
assert len(X_final) == len(y_final), "Features and target sizes do not match."

print("\n✓ Final dataset validation passed successfully.")


# 6. Train / Validation Split




### 6.1 Define Training and Validation Sets

We separate the model inputs from the target variable. The numerical features will be used as predictors, while the encoded `CLASS` variable will be used as the target.


In [ ]:
# ============================================
# 5.8 Define Training and Validation Sets
# ============================================

# Define final features and target
X = df_clean[features].copy()
y = df_clean[target].copy()

# Verify dimensions
print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Display the selected features
print("\nFeatures:")
print(X.columns.tolist())

# Display target classes
print("\nTarget classes:")
print(sorted(y.unique()))


### 6.2 Stratified Split

We split the dataset into training and validation sets using stratified sampling.

Stratification preserves the original class distribution in both subsets, which is important because the wildfire dataset contains more `no_fire` observations than `fire` observations.

We use 80% of the data for training and 20% for validation.


In [ ]:
from sklearn.model_selection import train_test_split

# Split the data using stratified sampling
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [ ]:
# Display training and validation shapes
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)


### 6.3 Feature Scaling

The numerical features have very different value ranges. NDVI is approximately between 0 and 1, while LST has values around 13,000–15,000.

We apply standardization so that each feature has a mean close to 0 and a standard deviation close to 1.

The scaler is fitted only on the training data and then applied to both the training and validation sets. This prevents information from the validation set from being used during training.


In [ ]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit on training data and transform training features
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation features using the same scaler
X_val_scaled = scaler.transform(X_val)


In [ ]:
# Check the mean and standard deviation of the scaled training features
print("Training mean:")
print(X_train_scaled.mean(axis=0))

print("\nTraining standard deviation:")
print(X_train_scaled.std(axis=0))


### 6.3 Verify Class Distribution

We compare the class distribution of the original dataset, training set, and validation set.

The proportions should remain approximately consistent because the data was split using stratified sampling.


In [ ]:
# Calculate class percentages for each dataset
original_distribution = y.value_counts(normalize=True) * 100
train_distribution = y_train.value_counts(normalize=True) * 100
val_distribution = y_val.value_counts(normalize=True) * 100

print("Original class distribution:")
print(original_distribution.round(2))

print("\nTraining class distribution:")
print(train_distribution.round(2))

print("\nValidation class distribution:")
print(val_distribution.round(2))


### 6.4 Record Dataset Sizes

In [ ]:
# Prepare class distribution percentages
distribution_df = pd.DataFrame({
    "Dataset": ["Original", "Original", "Training", "Training", "Validation", "Validation"],
    "Class": ["no_fire", "fire", "no_fire", "fire", "no_fire", "fire"],
    "Percentage": [
        original_distribution.get(0, 0),
        original_distribution.get(1, 0),
        train_distribution.get(0, 0),
        train_distribution.get(1, 0),
        val_distribution.get(0, 0),
        val_distribution.get(1, 0)
    ]
})

# Plot exact percentages
plt.figure(figsize=(9, 5))

ax = sns.barplot(
    data=distribution_df,
    x="Dataset",
    y="Percentage",
    hue="Class"
)

# Add percentage labels on top of each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3)

plt.title("Class Distribution: Original vs Training vs Validation")
plt.ylabel("Percentage (%)")
plt.xlabel("")
plt.ylim(0, 100)
plt.legend(title="Class")
plt.show()


In [ ]:
# Get the number of rows in each dataset
dataset_sizes = {
    "Original": len(y),
    "Training": len(y_train),
    "Validation": len(y_val)
}

# Plot the number of rows
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    x=list(dataset_sizes.keys()),
    y=list(dataset_sizes.values()),
    hue=list(dataset_sizes.keys()),
    legend=False,
    palette="Set2"
)

# Add exact row numbers on top of each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.title("Dataset Size: Original vs Training vs Validation")
plt.xlabel("Dataset")
plt.ylabel("Number of Rows")
plt.show()


# 7. Baseline Models

### 7.1 Baseline Model Selection

We train four different classification algorithms to establish baseline performance:

- Random Forest
- Decision Tree
- K-Nearest Neighbors (KNN)
- Support Vector Machine (SVM)

These models represent different learning approaches and will allow us to compare their performance on the wildfire classification problem before hyperparameter tuning.

Random Forest and Decision Tree are tree-based models, while KNN is distance-based and SVM is margin-based. This diversity provides a stronger baseline comparison than relying on a single algorithm.


### 7.2 Define Baseline Models

We define the four baseline classification models using their default configurations, with a fixed random state where applicable to improve reproducibility.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Define baseline models
models = {
    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "KNN": KNeighborsClassifier(),

    "SVM": SVC(
        probability=True,
        random_state=42
    )
}

# Display the models
for name, model in models.items():
    print(f"{name}:")
    print(model)
    print()


### 7.3 Build Training Pipelines

We create a reusable training function to train the four baseline models using a consistent workflow.

Tree-based models are trained using the original numerical features, while KNN and SVM use the standardized features because they are sensitive to feature scale.

The function returns the trained model so that it can be reused for validation predictions and evaluation.


In [ ]:
# Function to train a model

def train_model(model, X_train_data, y_train_data):
    """
    Train a classification model and return the fitted model.
    """

    model.fit(X_train_data, y_train_data)

    return model


In [ ]:
# Original features for tree-based models
X_train_tree = X_train
X_val_tree = X_val

# Scaled features for KNN and SVM
X_train_distance = X_train_scaled
X_val_distance = X_val_scaled


### 7.4 Train Baseline Models

We train the four selected baseline models using the appropriate training features.

Decision Tree and Random Forest use the original numerical features, while KNN and SVM use the standardized features.

All models are trained using the same training target to ensure a consistent comparison.

In [ ]:
# Train all baseline models

trained_models = {}

for name, model in models.items():

    # Use scaled features for KNN and SVM
    if name in ["KNN", "SVM"]:
        X_train_data = X_train_scaled
    else:
        X_train_data = X_train

    # Train the model
    trained_models[name] = train_model(
        model,
        X_train_data,
        y_train
    )

print("Baseline models trained successfully:")

for name in trained_models:
    print(f"- {name}")

In [ ]:
# Verify trained models

for name, model in trained_models.items():
    print(f"{name}: {type(model).__name__}")


### 7.5 Generate Validation Predictions

The trained baseline models are used to generate predictions on the validation dataset.

Random Forest and Decision Tree use the original validation features, while KNN and SVM use the standardized validation features.

The predictions will be used later to compare the performance of the four baseline models.


In [ ]:
# Generate validation predictions

validation_predictions = {}

for name, model in trained_models.items():

    # Use scaled features for KNN and SVM
    if name in ["KNN", "SVM"]:
        X_val_data = X_val_scaled
    else:
        X_val_data = X_val

    # Generate predictions
    validation_predictions[name] = model.predict(X_val_data)

print("Validation predictions generated successfully:")

for name in validation_predictions:
    print(f"- {name}")


In [ ]:
# Check prediction sizes

for name, predictions in validation_predictions.items():
    print(f"{name}: {len(predictions)} predictions")


### 7.6 Generate Prediction Probabilities

In addition to class predictions, we generate prediction probabilities for the validation observations.

The probability of the positive class (`fire = 1`) will be used later to calculate ROC-AUC and construct the ROC and Precision-Recall curves.


In [ ]:
# Generate validation probabilities for all baseline models

validation_probabilities = {}

for name, model in trained_models.items():

    # Use the appropriate validation features
    if name in ["KNN", "SVM"]:
        X_val_data = X_val_scaled
    else:
        X_val_data = X_val

    # Generate probability for the positive class (fire = 1)
    validation_probabilities[name] = model.predict_proba(X_val_data)[:, 1]

print("Validation probabilities generated successfully.")

for name, probabilities in validation_probabilities.items():
    print(f"{name}: {len(probabilities)} probabilities")


In [ ]:
# Display the first 10 predicted probabilities

for name, probabilities in validation_probabilities.items():
    print(f"\n{name}:")
    print(probabilities[:10])


# 8. Baseline Model Evaluation

### 8.1 Classification Metrics

We evaluate the four baseline models using accuracy, precision, recall, F1-score, and ROC-AUC.

These metrics provide a balanced view of model performance, particularly because the wildfire dataset contains more no_fire observations than fire observations.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Store baseline evaluation results
baseline_results = []

for name in trained_models:

    y_pred = validation_predictions[name]
    y_prob = validation_probabilities[name]

    baseline_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, y_prob)
    })

# Create comparison DataFrame
baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df


In [ ]:
# Plot baseline model performance

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC"
]

plot_data = baseline_results_df.set_index("Model")[metrics]

ax = plot_data.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Baseline Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.grid(axis="y", alpha=0.3)

plt.show()


### 8.2 Confusion Matrix Analysis

We analyze the confusion matrices of the four baseline models to understand their classification errors.

Particular attention is given to false negatives because failing to identify an actual wildfire is an important error in a wildfire detection problem.


In [ ]:
from sklearn.metrics import confusion_matrix

# Create confusion matrices for all baseline models
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, (name, predictions) in zip(axes.ravel(), validation_predictions.items()):

    cm = confusion_matrix(y_val, predictions)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["no_fire", "fire"],
        yticklabels=["no_fire", "fire"]
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Baseline Models — Confusion Matrices", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Display TP, TN, FP, and FN for each model

confusion_results = []

for name, predictions in validation_predictions.items():

    tn, fp, fn, tp = confusion_matrix(y_val, predictions).ravel()

    confusion_results.append({
        "Model": name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

confusion_results_df = pd.DataFrame(confusion_results)

confusion_results_df


### 8.3 ROC-AUC, ROC Curve, and Precision-Recall Curve

We compare the baseline models using ROC-AUC and Precision-Recall analysis.

The ROC curve evaluates the trade-off between the true positive rate and false positive rate across classification thresholds.

The Precision-Recall curve is particularly useful for this dataset because the target classes are imbalanced and the `fire` class is the minority class.


In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC curves
for name, probabilities in validation_probabilities.items():

    fpr, tpr, _ = roc_curve(y_val, probabilities)
    auc_score = roc_auc_score(y_val, probabilities)

    axes[0].plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc_score:.3f})"
    )

axes[0].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Random"
)

axes[0].set_title("ROC Curves")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()
axes[0].grid(alpha=0.3)


# Precision-Recall curves
for name, probabilities in validation_probabilities.items():

    precision, recall, _ = precision_recall_curve(
        y_val,
        probabilities
    )

    ap_score = average_precision_score(
        y_val,
        probabilities
    )

    axes[1].plot(
        recall,
        precision,
        label=f"{name} (AP = {ap_score:.3f})"
    )

axes[1].set_title("Precision-Recall Curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(
    "Baseline Models — ROC and Precision-Recall Analysis",
    fontsize=15
)

plt.tight_layout()
plt.show()


### 8.4 Baseline Model Comparison

The baseline models are compared using accuracy, precision, recall, F1-score, and ROC-AUC.

Because the dataset is imbalanced and wildfire detection is the main objective, F1-score, recall, and ROC-AUC are given particular attention when comparing the models.


In [ ]:
# Sort baseline models by F1-score
baseline_comparison = baseline_results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

# Display the comparison
baseline_comparison.round(4)


In [ ]:
# Visualize baseline F1-score and ROC-AUC

comparison_plot = baseline_comparison.set_index("Model")[
    ["F1 Score", "ROC-AUC"]
]

ax = comparison_plot.plot(
    kind="bar",
    figsize=(10, 5),
    color=["steelblue", "darkorange"]
)

plt.title("Baseline Model Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend()
plt.grid(axis="y", alpha=0.3)

# Add values to bars
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)

plt.show()


In [ ]:
# Identify the best baseline model according to F1-score
best_baseline_model = baseline_comparison.iloc[0]["Model"]
best_baseline_f1 = baseline_comparison.iloc[0]["F1 Score"]

print(f"Best baseline model by F1-score: {best_baseline_model}")
print(f"F1-score: {best_baseline_f1:.4f}")


### 8.5 Baseline Error Analysis

We analyze the classification errors produced by the baseline models.

Special attention is given to false negatives, where an actual fire observation is incorrectly classified as no_fire. These errors are particularly important in a wildfire detection context.


In [ ]:
# Analyze classification errors for all baseline models

error_analysis = []

for name, predictions in validation_predictions.items():

    tn, fp, fn, tp = confusion_matrix(y_val, predictions).ravel()

    error_analysis.append({
        "Model": name,
        "False Positives": fp,
        "False Negatives": fn,
        "Total Errors": fp + fn
    })

error_analysis_df = pd.DataFrame(error_analysis)

error_analysis_df


In [ ]:
# Visualize false positives and false negatives

ax = error_analysis_df.set_index("Model")[
    ["False Positives", "False Negatives"]
].plot(
    kind="bar",
    figsize=(10, 5),
    color=["orange", "red"]
)

plt.title("Baseline Model Classification Errors")
plt.xlabel("Model")
plt.ylabel("Number of Errors")
plt.xticks(rotation=0)
plt.legend()
plt.grid(axis="y", alpha=0.3)

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.show()


In [ ]:
# Compare false negatives
best_fire_detection_model = error_analysis_df.loc[
    error_analysis_df["False Negatives"].idxmin()
]

print(
    f"Fewest missed fires: "
    f"{best_fire_detection_model['Model']}"
)

print(
    f"False negatives: "
    f"{best_fire_detection_model['False Negatives']}"
)


# 9. Hyperparameter Tuning

### 9.1 Define Parameter Grids

We define a set of hyperparameters for each baseline model that will be evaluated using GridSearchCV.

The goal is to find better-performing model configurations while avoiding unnecessary combinations.

The tuning process will be performed separately for Random Forest, Decision Tree, KNN, and SVM.


In [ ]:
# Define hyperparameter grids for each model

param_grids = {

    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    },

    "Decision Tree": {
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    },

    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"],
        "p": [1, 2]
    },

    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale", "auto"]
    }
}


In [ ]:
from sklearn.model_selection import ParameterGrid

# Count parameter combinations
for name, grid in param_grids.items():
    combinations = len(list(ParameterGrid(grid)))
    print(f"{name}: {combinations} combinations")


### 9.2 GridSearchCV Configuration

GridSearchCV evaluates different hyperparameter combinations using cross-validation.

We use 5-fold stratified cross-validation and F1-score as the primary optimization metric. F1-score is selected because the dataset is imbalanced and both precision and recall are important for wildfire classification.

The best configuration will be selected based on the mean validation F1-score across the cross-validation folds.


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Create stratified cross-validation
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# GridSearchCV settings
gridsearch_settings = {
    "cv": cv_strategy,
    "scoring": "f1",
    "n_jobs": -1,
    "refit": True
}

print("GridSearchCV configuration:")
print("Cross-validation folds: 5")
print("Scoring metric: F1-score")
print("Parallel processing: enabled")
print("Refit best model: True")


### 9.3 Tune Baseline Models

We perform hyperparameter tuning for the four baseline models using GridSearchCV.

Each model is evaluated using 5-fold stratified cross-validation, with F1-score as the optimization metric.

The best hyperparameter configuration for each model is retained for later validation and comparison.


In [ ]:
# Run GridSearchCV for all four models

grid_searches = {}
tuned_models = {}

for name, model in models.items():

    # Select the appropriate training data
    if name in ["KNN", "SVM"]:
        X_train_data = X_train_scaled
    else:
        X_train_data = X_train

    print(f"\nTuning {name}...")

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        **gridsearch_settings
    )

    # Run hyperparameter search
    grid_search.fit(X_train_data, y_train)

    # Store GridSearchCV object
    grid_searches[name] = grid_search

    # Store best fitted model
    tuned_models[name] = grid_search.best_estimator_

    print(f"Best F1-score: {grid_search.best_score_:.4f}")
    print(f"Best parameters: {grid_search.best_params_}")


### 9.4 Best Hyperparameters

We record the best hyperparameter configuration identified by GridSearchCV for each baseline model.

The best configuration is selected according to the mean cross-validation F1-score obtained during the tuning process.


In [ ]:
# Create a table of the best hyperparameters

best_parameters = []

for name, grid_search in grid_searches.items():
    best_parameters.append({
        "Model": name,
        "Best CV F1": grid_search.best_score_,
        "Best Parameters": grid_search.best_params_
    })

best_parameters_df = pd.DataFrame(best_parameters)

best_parameters_df


### 9.5 Train Tuned Models

GridSearchCV has already refitted the best configuration for each model using the complete training dataset.

We now use these tuned estimators to generate predictions and probabilities on the same validation set used for the baseline evaluation.


In [ ]:
# Generate validation predictions and probabilities for tuned models

tuned_predictions = {}
tuned_probabilities = {}

for name, model in tuned_models.items():

    # Select the appropriate validation data
    if name in ["KNN", "SVM"]:
        X_val_data = X_val_scaled
    else:
        X_val_data = X_val

    # Generate class predictions
    tuned_predictions[name] = model.predict(X_val_data)

    # Generate probability of the fire class
    tuned_probabilities[name] = model.predict_proba(X_val_data)[:, 1]

print("Tuned model predictions generated successfully.")

for name in tuned_models:
    print(f"- {name}")


In [ ]:
# Check the number of predictions and probabilities

for name in tuned_models:
    print(
        f"{name}: "
        f"{len(tuned_predictions[name])} predictions, "
        f"{len(tuned_probabilities[name])} probabilities"
    )


### 9.6 Evaluate Tuned Models

The tuned models are evaluated on the same validation dataset used for the baseline models.

Accuracy, Precision, Recall, F1-score, and ROC-AUC are calculated for each tuned model.

The tuned results are then compared with the baseline results to determine whether hyperparameter tuning improved model performance.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Evaluate tuned models

tuned_results = []

for name in tuned_models:

    y_pred = tuned_predictions[name]
    y_prob = tuned_probabilities[name]

    tuned_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, y_prob)
    })

tuned_results_df = pd.DataFrame(tuned_results)

tuned_results_df.round(4)


## 10 Before vs After Tuning

The accuracy of the four baseline models is compared with their corresponding tuned versions.

This comparison shows whether hyperparameter tuning improved the overall classification accuracy of each model.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create accuracy comparison
accuracy_comparison = pd.DataFrame({
    "Before Tuning": baseline_results_df.set_index("Model")["Accuracy"],
    "After Tuning": tuned_results_df.set_index("Model")["Accuracy"]
})

# Display accuracy values
display(accuracy_comparison.round(4))


# -----------------------------
# Plot Accuracy Before vs After
# -----------------------------

model_names = accuracy_comparison.index

scores_before = accuracy_comparison["Before Tuning"].values
scores_after = accuracy_comparison["After Tuning"].values

x = np.arange(len(model_names))
width = 0.35

plt.figure(figsize=(10, 6))

bars_before = plt.bar(
    x - width / 2,
    scores_before,
    width,
    label="Before Tuning",
    color="steelblue"
)

bars_after = plt.bar(
    x + width / 2,
    scores_after,
    width,
    label="After Tuning",
    color="seagreen"
)

# Add accuracy values
for bar in bars_before:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.3f}",
        ha="center"
    )

for bar in bars_after:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.3f}",
        ha="center"
    )

plt.xticks(x, model_names)
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.title("Model Accuracy Before and After Hyperparameter Tuning")
plt.ylim(0, 1)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Prepare data for visualization

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC"
]

plot_data = tuned_results_df.set_index("Model")[metrics]

# Plot
ax = plot_data.plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8
)

plt.title("Tuned Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Metrics")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


### 11. Save the Best Model

The best tuned model is selected based on the highest validation accuracy.

The selected model is then saved to disk so that it can be reused later without retraining.


#### 11.1 Save Best Model Pipeline

In [ ]:
# Find the best tuned model based on accuracy

best_model_row = tuned_results_df.loc[
    tuned_results_df["Accuracy"].idxmax()
]

best_model_name = best_model_row["Model"]
best_model = tuned_models[best_model_name]

print("Best Model:", best_model_name)
print(f"Accuracy: {best_model_row['Accuracy']:.4f}")


In [ ]:
import joblib

# Find the best tuned model based on validation accuracy
best_model_row = tuned_results_df.loc[
    tuned_results_df["Accuracy"].idxmax()
]

best_model_name = best_model_row["Model"]
best_model = tuned_models[best_model_name]
best_accuracy = best_model_row["Accuracy"]

print("Best Model:", best_model_name)
print(f"Validation Accuracy: {best_accuracy:.4f}")

# Save the best model
model_filename = "best_wildfire_model.pkl"

joblib.dump(best_model, model_filename)

print(f"\nBest model saved as: {model_filename}")


### 11.2 Verify Saved Model

The saved model is loaded from disk to verify that the file was created correctly and that the model can be restored successfully.


In [ ]:
import joblib

# Load the saved best model
loaded_model = joblib.load("best_wildfire_model.pkl")

print("Saved model loaded successfully.")
print("Model:", type(loaded_model).__name__)


### 11.4 Test Saved Model

The saved model is tested on the validation dataset to confirm that it produces the same predictions and accuracy as the original tuned model.


In [ ]:
from sklearn.metrics import accuracy_score

# Select the appropriate validation features
if best_model_name in ["KNN", "SVM"]:
    X_val_test = X_val_scaled
else:
    X_val_test = X_val

# Generate predictions using the saved model
saved_predictions = loaded_model.predict(X_val_test)

# Calculate accuracy
saved_accuracy = accuracy_score(y_val, saved_predictions)

print("Saved Model Test Results")
print("------------------------")
print("Model:", best_model_name)
print(f"Accuracy: {saved_accuracy:.4f}")


In [ ]:
# Compare original and saved model predictions
predictions_match = np.array_equal(
    best_model.predict(X_val_test),
    saved_predictions
)

print("Predictions match:", predictions_match)


In [ ]:
# Enter new environmental values
ndvi = 0.25
lst = 35.0
burned_area = 12.0

# Create input DataFrame
new_data = pd.DataFrame({
    "NDVI": [ndvi],
    "LST": [lst],
    "BURNED_AREA": [burned_area]
})

new_data


In [ ]:
# Generate prediction
prediction = best_model.predict(new_data)[0]

# Generate probability
probabilities = best_model.predict_proba(new_data)[0]

fire_probability = probabilities[1]
no_fire_probability = probabilities[0]

# Convert numerical prediction to class label
predicted_class = "fire" if prediction == 1 else "no_fire"

print("Wildfire Prediction")
print("-------------------")
print(f"NDVI: {ndvi}")
print(f"LST: {lst}")
print(f"BURNED_AREA: {burned_area}")
print()
print(f"Predicted Class: {predicted_class}")
print(f"Fire Probability: {fire_probability:.2%}")
print(f"No-Fire Probability: {no_fire_probability:.2%}")


## 13. Conclusion

### What We Built

We developed a complete machine learning workflow for wildfire classification using three numerical features: **NDVI, LST, and BURNED_AREA**.

The workflow included data preprocessing, missing-value interpolation, duplicate removal, feature transformation, stratified train-validation splitting, baseline model training, model evaluation, hyperparameter tuning using `GridSearchCV`, and final model selection.

Four classification algorithms were evaluated:

- Random Forest
- Decision Tree
- KNN
- SVM

### Baseline Findings

The four models were first trained using their baseline configurations and evaluated on the validation dataset.

This provided a reference point for comparing model performance before hyperparameter tuning.

### Tuning Results

`GridSearchCV` was used to search for improved hyperparameter configurations using **5-fold stratified cross-validation** and **F1-score** as the optimization criterion.

After tuning, the models were evaluated again on the same validation dataset. Their accuracy scores were compared with the baseline results to determine whether hyperparameter optimization improved performance.

### Best Model

The final model was selected based on the **highest validation accuracy**.

The **Random Forest** model achieved the highest validation accuracy of **80.54%** and was therefore selected as the final model.

The tuned Random Forest model was saved as `best_wildfire_model.pkl` and successfully loaded and tested to verify that the saved model produces the expected predictions.

### Next Steps

The next stage is to move the developed machine learning model from the research notebook into a production-ready wildfire prediction system.

The main steps would include:

- **Prepare the final model:** Save the selected Random Forest model together with the preprocessing steps required to transform new input data in exactly the same way as the training data.

- **Build a prediction pipeline:** Create a complete pipeline that accepts new environmental observations such as NDVI, LST, and BURNED_AREA and returns a wildfire prediction and prediction probability.

- **Develop a backend API:** Deploy the trained model through an API that allows other applications to send environmental data and receive wildfire predictions.

- **Create a user interface:** Develop a web or dashboard application where users can provide input data and view the predicted wildfire class and probability.

- **Connect real-world data sources:** Integrate satellite, meteorological, and other environmental data sources so that predictions can be generated from continuously updated observations rather than manually entered data.

- **Deploy the system:** Host the model and application on a cloud or production server so that the prediction service can be accessed remotely.

- **Monitor model performance:** Continuously monitor prediction accuracy, input data quality, and changes in environmental conditions to detect when the model needs to be retrained.

- **Retrain and update the model:** As new wildfire observations become available, periodically retrain the model to improve its performance and adapt to changing environmental patterns.

- **Final production system:** The final product could provide near-real-time wildfire risk predictions through a web dashboard or API, helping users identify areas with a higher likelihood of wildfire occurrence.

